In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoImageProcessor, AutoModelForImageClassification, TrainingArguments, Trainer
from datasets import load_dataset
import timm

# --- Configuration ---
TEACHER_MODEL_PATH = os.path.expanduser("/media/rajendraprasath-m/New Volume/Projects/Final Year Project/Model/vit_final_model")
DATASET_PATH = "/media/rajendraprasath-m/New Volume/Projects/Final Year Project/Data/Video/Extracted_Frames"
STUDENT_MODEL_NAME = "facebook/deit-tiny-patch16-224"  # A very popular TinyViT variant
OPTIMIZED_DIR = os.path.expanduser("/media/rajendraprasath-m/New Volume/Projects/Final Year Project/Model/tiny_vit_distilled")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
from transformers import Trainer
import torch
import torch.nn.functional as F

class DistillationTrainer(Trainer):
    def __init__(self, teacher_model=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher = teacher_model
        self.teacher.eval()

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")

        # Student forward
        outputs_student = model(**inputs)
        student_logits = outputs_student.logits

        # Teacher forward (no grad)
        with torch.no_grad():
            outputs_teacher = self.teacher(**inputs)
            teacher_logits = outputs_teacher.logits

        # Losses
        loss_ce = F.cross_entropy(student_logits, labels)

        loss_kd = F.kl_div(
            F.log_softmax(student_logits / 2.0, dim=-1),
            F.softmax(teacher_logits / 2.0, dim=-1),
            reduction="batchmean"
        ) * (2.0 ** 2)

        loss = 0.5 * loss_ce + 0.5 * loss_kd

        return (loss, outputs_student) if return_outputs else loss

In [3]:
# 1. Load Dataset
dataset = load_dataset("imagefolder", data_dir=DATASET_PATH)

# --- SAFETY CHECK ---
print(f"Dataset Columns: {dataset['train'].column_names}")
# --------------------

# Get the labels
labels = dataset["train"].features["label"].names
num_labels = len(labels)
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in label2id.items()}

# 2. Load Processors
image_processor = AutoImageProcessor.from_pretrained(TEACHER_MODEL_PATH)

# 3. Load Teacher and Student
teacher_model = AutoModelForImageClassification.from_pretrained(TEACHER_MODEL_PATH).to(device)
student_model = AutoModelForImageClassification.from_pretrained(
    STUDENT_MODEL_NAME,
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True
).to(device)



Resolving data files:   0%|          | 0/54519 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21937 [00:00<?, ?it/s]

Dataset Columns: ['image', 'label']


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: facebook/deit-tiny-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 192]) vs model:torch.Size([2, 192])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


In [11]:
def transform(examples):
    inputs = image_processor(examples["image"])
    inputs["labels"] = examples["label"]
    return inputs

# Apply the transformation to the splits
dataset["train"] = dataset["train"].with_transform(transform)
dataset["test"] = dataset["test"].with_transform(transform)
 
print(dataset["train"].column_names)
print(f"[SUCCESS] Teacher and Student ready. Labels: {id2label}")   

['image', 'label']
[SUCCESS] Teacher and Student ready. Labels: {'NonViolence': 0, 'Violence': 1}


In [21]:
training_args = TrainingArguments(
    output_dir=OPTIMIZED_DIR,
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=200,
    save_total_limit=1,
    report_to="none",
    fp16=True,
    remove_unused_columns=False
)

In [22]:
distill_trainer = DistillationTrainer(
    teacher_model=teacher_model,
    model=student_model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
)


In [23]:
dataset["train"] = dataset["train"].shuffle(seed=42).select(range(10000))
dataset["test"] = dataset["test"].select(range(2000))

In [24]:
for param in teacher_model.parameters():
    param.requires_grad = False

In [25]:
print("[INFO] Starting Knowledge Distillation...")
distill_trainer.train()


[INFO] Starting Knowledge Distillation...


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
distill_trainer.save_model(OPTIMIZED_DIR)
print(f"[SUCCESS] Optimized TinyViT saved at {OPTIMIZED_DIR}")

In [ ]:
import torch.nn.utils.prune as prune

# Apply 20% unstructured pruning to the linear layers of the student
for name, module in student_model.named_modules():
    if isinstance(module, torch.nn.Linear):
        prune.l1_unstructured(module, name='weight', amount=0.2)
        prune.remove(module, 'weight') # Make the pruning permanent

# Save the final pruned model
torch.save(student_model.state_dict(), os.path.join(OPTIMIZED_DIR, "pruned_tiny_vit.pt"))
print("[SUCCESS] Model pruned by 20% and saved.")